In [1]:
import fiftyone.zoo as foz

dataset = foz.load_zoo_dataset(
    "coco-2017",
    #splits=["train", "validation", "test"],
    splits=["train"],
    label_types=["detections"],
    classes=["chair"],
    max_samples=1000,
)


Found annotations at '/home/freeze/fiftyone/coco-2017/raw/instances_train2017.json'
Sufficient images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'coco-2017-train-1000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [5]:
print(dataset)

Name:        coco-2017-train-1000
Media type:  image
Num samples: 1000
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)


In [2]:
dataset.compute_metadata()

In [4]:
import torch
from ultralytics import YOLO

# Run the model on GPU if it is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the YOLO model (not just the path)
model = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/doors/best.pt")
model.to(device)

print("Model ready")

Model ready


In [7]:
from PIL import Image
import numpy as np
import fiftyone as fo

# Get class list
classes = dataset.default_classes

# Add predictions to samples
with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        # Load image as PIL Image (don't convert to tensor)
        image = Image.open(sample.filepath)
        
        # Perform inference directly on PIL image
        results = model(image)
        
        # Extract predictions from results
        result = results[0]
        boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy().astype(int)
        
        # Get image dimensions
        w, h = image.size
        
        # Convert detections to FiftyOne format
        detections = []
        for label, score, box in zip(labels, scores, boxes):
            # Convert to [top-left-x, top-left-y, width, height]
            # in relative coordinates in [0, 1] x [0, 1]
            x1, y1, x2, y2 = box
            rel_box = [x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h]

            detections.append(
                fo.Detection(
                    label=model.names[label],  # Use model's class names
                    bounding_box=rel_box,
                    confidence=score
                )
            )

        # Save predictions to dataset
        sample["predictions"] = fo.Detections(detections=detections)
        sample.save()

   0% ||--------------|    0/1000 [10.2ms elapsed, ? remaining, ? samples/s] 

                                                                             
0: 480x640 (no detections), 7.6ms
Speed: 18.3ms preprocess, 7.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 640x416 (no detections), 7.9ms
Speed: 0.9ms preprocess, 7.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 416)

0: 480x640 (no detections), 7.7ms
Speed: 0.8ms preprocess, 7.7ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 640x448 (no detections), 7.0ms
Speed: 0.8ms preprocess, 7.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)
                                                                                     
0: 448x640 (no detections), 7.6ms
Speed: 0.7ms preprocess, 7.6ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)

0: 480x640 (no detections), 7.5ms
Speed: 0.7ms preprocess, 7.5ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 6.6ms
Speed: 1.2ms pre

In [9]:


# Filter samples that have "Door" predictions
door_predictions_view = dataset.filter_labels("predictions", fo.ViewField("label") == "Door")

# Get the sample IDs that contain Door predictions
door_sample_ids = list(door_predictions_view.values("id"))

# Delete these samples from the dataset
dataset.delete_samples(door_sample_ids)

print(f"Deleted {len(door_sample_ids)} samples with Door predictions")

# Save the changes
dataset.save()

Deleted 107 samples with Door predictions


In [ ]:
# More efficient approach using bulk updates
import numpy as np

# Create split labels
total = len(dataset)
splits = (["train"] * int(0.7 * total) + 
          ["val"] * int(0.2 * total) + 
          ["test"] * (total - int(0.7 * total) - int(0.2 * total)))

# Shuffle splits
np.random.shuffle(splits)

# Assign splits to samples
dataset.set_values("split", splits)

print(f"Split created: {splits.count('train')} train, {splits.count('val')} val, {splits.count('test')} test")

Split created: 625 train, 178 val, 90 test


In [ ]:

import fiftyone.utils.random as four

four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(dataset.count_sample_tags())

{'val': 89, 'train': 893, 'test': 179}


In [21]:
# See unique values in the split field
print("Split values:", dataset.distinct("split"))

# Count samples per split
split_counts = dataset.count_values("split")
print("Split counts:", split_counts)

Split values: ['test', 'train', 'val']
Split counts: {'val': 178, 'train': 625, 'test': 90}


In [15]:
# Visualize the dataset in the FiftyOne App
import fiftyone as fo
session = fo.launch_app(dataset)

In [ ]:


export_dir = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test"
#label_field = "ground_truth"  # for example

# The splits to export
splits = ["train", "test","val"]

# All splits must use the same classes list
classes = ["chair"]

# Export the splits
for split in splits:
    split_view = dataset.match_tags(split)
    split_view.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=label_field,
        split=split,
        classes=classes,
    )

Directory '/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test' already exists; export will be merged with existing files
   0% ||----------------|   0/893 [826.8ms elapsed, ? remaining, ? samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'fork' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'knife' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'pizza' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'dining table' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'person' not in provided classes
  warnings.warn(msg)
/home/freeze/

   4% |\----------------|  39/893 [2.0s elapsed, 44.8s remaining, 19.1 samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'cake' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'bottle' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'refrigerator' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'wine glass' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'bowl' not in provided classes
  warnings.warn(msg)
/home/fre

   9% |█/---------------|  76/893 [2.3s elapsed, 24.2s remaining, 33.7 samples/s] 

...

  71% |████████████|----| 633/893 [5.0s elapsed, 1.5s remaining, 203.3 samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'traffic light' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'sheep' not in provided classes
  warnings.warn(msg)


  81% |█████████████|---| 721/893 [5.5s elapsed, 904.2ms remaining, 212.6 samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'fire hydrant' not in provided classes
  warnings.warn(msg)


 100% |█████████████████| 893/893 [6.3s elapsed, 0s remaining, 193.7 samples/s]      
Directory '/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test' already exists; export will be merged with existing files
 100% |█████████████████| 179/179 [1.8s elapsed, 0s remaining, 188.6 samples/s]     
Directory '/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test' already exists; export will be merged with existing files
 100% |███████████████████| 89/89 [1.4s elapsed, 0s remaining, 65.3 samples/s]      
